# Document QA System (RAG)

A Retrieval-Augmented Generation system that answers questions about your documents.

**Use cases:** Company handbook QA, research paper assistant, course materials search.

### Architecture

```
Upload Documents
      ↓
Text Extraction (PyPDF / TextLoader)
      ↓
Chunking (RecursiveCharacterTextSplitter)
      ↓
Embedding API (HuggingFace Inference API)
      ↓
Vector Store (InMemoryVectorStore)
      ↓
User Question → Embed → Vector Search → Top-K Chunks
      ↓
LLM API (Qwen2.5-7B-Instruct via HuggingFace)
      ↓
Answer with Source Citations
```

**No local LLM hosting required** — all inference runs via the free HuggingFace Inference API.

## 1. Install Dependencies

In [1]:
import langchain_core
print(f"langchain-core: {langchain_core.__version__}")

langchain-core: 1.2.19


## 2. Imports

In [2]:
import os
import warnings
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from huggingface_hub import InferenceClient


warnings.filterwarnings("ignore", category=DeprecationWarning)

/opt/homebrew/Caskroom/miniconda/base/envs/venv_nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configuration

Get a free HuggingFace API token at https://huggingface.co/settings/tokens

In [3]:
# --- API Token ---
HF_TOKEN = os.environ.get("HUGGINGFACE_API_KEY", "")

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Enter your HuggingFace API token: ")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
# os.environ["OMP_NUM_THREADS"] = "1"

# --- Models ---
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
# Alternatives if the above is unavailable:
# LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
# LLM_MODEL = "microsoft/Phi-3.5-mini-instruct"

# --- Chunking ---
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# --- Retrieval ---
NUM_RETRIEVED_DOCS = 4

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model:       {LLM_MODEL}")
print(f"Chunk size:      {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP}")
print(f"Retrieval top-k: {NUM_RETRIEVED_DOCS}")

Embedding model: sentence-transformers/all-MiniLM-L6-v2
LLM model:       Qwen/Qwen2.5-7B-Instruct
Chunk size:      1000 chars, overlap: 200
Retrieval top-k: 4


## 4. Document Loading

**Supported formats:** PDF (`.pdf`), plain text (`.txt`), markdown (`.md`)

- **Google Colab:** A file upload widget will appear.
- **Jupyter/local:** Place your files in the `uploaded_docs/` folder, then run the cell.

In [4]:
DOCS_DIR = Path("./uploaded_docs")
DOCS_DIR.mkdir(exist_ok=True)


def upload_documents():
    """Upload documents via Colab widget or list files in the docs directory."""
    print(f"Place your PDF/TXT/MD files in: {DOCS_DIR.resolve()}")
    print("Then re-run this cell.")

    files = [f for f in DOCS_DIR.iterdir() if f.is_file()]
    print(f"\nFound {len(files)} file(s): {[f.name for f in files]}")
    return files


def load_documents(file_paths):
    """Load documents from file paths. Returns a list of langchain Documents."""
    all_docs = []

    for path in file_paths:
        path = Path(path)
        try:
            if path.suffix.lower() == ".pdf":
                loader = PyPDFLoader(str(path))
                docs = loader.load()
                print(f"  PDF:  {path.name} ({len(docs)} pages)")
            elif path.suffix.lower() in (".txt", ".md"):
                loader = TextLoader(str(path), encoding="utf-8")
                docs = loader.load()
                print(f"  Text: {path.name}")
            else:
                print(f"  Skipping unsupported file: {path.name}")
                continue
            all_docs.extend(docs)
        except Exception as e:
            print(f"  Error loading {path.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs


file_paths = upload_documents()
documents = load_documents(file_paths)

Place your PDF/TXT/MD files in: /Users/rafi/Code/python/nlp_projects/notebooks/uploaded_docs
Then re-run this cell.

Found 1 file(s): ['Abdullah_Al_Rafi.pdf']
  PDF:  Abdullah_Al_Rafi.pdf (2 pages)

Total documents loaded: 2


## 5. Text Chunking

Documents are split into smaller chunks so we can retrieve only the most relevant pieces for each question. Overlap between chunks preserves context at boundaries.

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(documents)

print(f"Documents: {len(documents)} → Chunks: {len(chunks)}")
print(f"Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP}")

if chunks:
    print(f"\n--- First chunk preview ---")
    print(f"Content ({len(chunks[0].page_content)} chars):")
    print(chunks[0].page_content[:300] + "...")
    print(f"Metadata: {chunks[0].metadata}")

Documents: 2 → Chunks: 5
Chunk size: 1000 chars, overlap: 200

--- First chunk preview ---
Content (981 chars):
Abdullah Al Raﬁ
+49 163 4005423 | 16(b) | abdullah34alraﬁ@gmail.com | linkedin.com/in/abdullahalraﬁ | github.com/alraﬁabdullah |
www.abdullahalraﬁ.com
SUMMARY
Machine Learning Engineer with 4+ years of experience building and deploying large-scale NLP and computer vision
systems in ﬁntech environmen...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-02T00:56:07+00:00', 'author': '', 'keywords': '', 'moddate': '2026-03-02T00:56:07+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'uploaded_docs/Abdullah_Al_Rafi.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


## 6. Embedding & Vector Store

Each chunk is converted to a vector using `all-MiniLM-L6-v2` via the HuggingFace API, then stored in `InMemoryVectorStore` for similarity search (stable on Apple Silicon).

In [6]:
embeddings = HuggingFaceEndpointEmbeddings(
    model=EMBEDDING_MODEL,
    huggingfacehub_api_token=HF_TOKEN,
)

# Verify the embedding model works
test_embedding = embeddings.embed_query("test")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Embedding dim:   {len(test_embedding)}")

# Build in-memory vector store from all chunks
print(f"\nEmbedding and indexing {len(chunks)} chunks...")
vectorstore = InMemoryVectorStore.from_documents(chunks, embedding=embeddings)
print("InMemoryVectorStore ready")

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dim:   384

Embedding and indexing 5 chunks...
InMemoryVectorStore ready


## 7. RAG Chain Setup

The vectorstore retrieves the top-K most relevant chunks, then the LLM generates an answer conditioned on those chunks.

In [7]:
client = InferenceClient(model=LLM_MODEL, token=HF_TOKEN)

# Quick test
print(f"LLM: {LLM_MODEL}")
test_response = client.chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=64,
)
print(f"LLM test: {test_response.choices[0].message.content}")

# Prompt template for RAG
QA_PROMPT = PromptTemplate.from_template(
    """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {question}

Helpful Answer:"""
)

print("\nRetriever + LLM ready!")

LLM: Qwen/Qwen2.5-7B-Instruct
LLM test: Hello there!

Retriever + LLM ready!


## 8. Ask Questions

In [8]:
def ask_question(question, show_sources=True):
    """Ask a question about the loaded documents."""
    required_objects = ["vectorstore", "client", "QA_PROMPT"]
    missing = [name for name in required_objects if name not in globals()]
    if missing:
        raise RuntimeError(
            f"Missing setup objects: {missing}. Run sections 2-7 (or load index in section 10) first."
        )

    print(f"Q: {question}")
    print("-" * 60)

    docs = vectorstore.similarity_search(question, k=NUM_RETRIEVED_DOCS)

    # Build prompt with context
    context = "\n\n".join(doc.page_content for doc in docs)
    prompt = QA_PROMPT.format(context=context, question=question)

    # Generate answer via chat completion
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.7,
    )
    answer = response.choices[0].message.content.strip()
    print(f"\nA: {answer}")

    if show_sources and docs:
        print(f"\n{'=' * 60}")
        print(f"Sources ({len(docs)} chunks):")
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source", "unknown")
            page = doc.metadata.get("page", "N/A")
            print(f"\n  [{i}] {Path(source).name}, page {page}")
            print(f"      {doc.page_content[:150]}...")

    return answer

In [9]:
answer = ask_question("What is the main topic of the documents?")
answer

Q: What is the main topic of the documents?
------------------------------------------------------------

A: The main topic of the documents appears to be the professional experience and skills of Abdullah Al Raﬁ, specifically focusing on his roles as an ML Engineer and Data Scientist, his projects, and his skills in machine learning, natural language processing, and computer vision.

Sources (4 chunks):

  [1] Abdullah_Al_Rafi.pdf, page 0
      Data Science, bKash Limited Dhaka, Bangladesh
• Migrated and productionized Customer Churn, Business Anomaly Detection, and Customer Lifetime Value pi...

  [2] Abdullah_Al_Rafi.pdf, page 0
      Abdullah Al Raﬁ
+49 163 4005423 | 16(b) | abdullah34alraﬁ@gmail.com | linkedin.com/in/abdullahalraﬁ | github.com/alraﬁabdullah |
www.abdullahalraﬁ.com...

  [3] Abdullah_Al_Rafi.pdf, page 0
      • MLOps - Nvidia Triton, ONNXRuntime, Docker
• Cloud - AWS, GCP , Digital Ocean, Railway, GitHub Actions, Jenkins
• Web - Django, FastAPI, Flask, Reac...

  [

'The main topic of the documents appears to be the professional experience and skills of Abdullah Al Raﬁ, specifically focusing on his roles as an ML Engineer and Data Scientist, his projects, and his skills in machine learning, natural language processing, and computer vision.'

In [10]:
# Debug: isolate which step crashes the kernel
import sys

print("Step 1: Testing embedding API...")
sys.stdout.flush()
test_vec = embeddings.embed_query("test query")
print(f"  OK — got {len(test_vec)}-dim vector")

print("Step 2: Testing vectorstore search...")
sys.stdout.flush()
docs = vectorstore.similarity_search("test query", k=2)
print(f"  OK — got {len(docs)} docs")

print("Step 3: Testing LLM call...")
sys.stdout.flush()
resp = client.chat_completion(
    messages=[{"role": "user", "content": "Say hi"}],
    max_tokens=32,
    temperature=0,
)
print(f"  OK — {resp.choices[0].message.content}")

print("Step 4: Full ask_question...")
sys.stdout.flush()
ask_question("What is this document about?")

Step 1: Testing embedding API...
  OK — got 384-dim vector
Step 2: Testing vectorstore search...
  OK — got 2 docs
Step 3: Testing LLM call...
  OK — Hi there! How can I assist you today?
Step 4: Full ask_question...
Q: What is this document about?
------------------------------------------------------------

A: This document appears to be a resume or CV for Abdullah Al Raﬁ, detailing his professional experience, education, skills, and achievements in the field of machine learning and data science.

Sources (4 chunks):

  [1] Abdullah_Al_Rafi.pdf, page 0
      Data Science, bKash Limited Dhaka, Bangladesh
• Migrated and productionized Customer Churn, Business Anomaly Detection, and Customer Lifetime Value pi...

  [2] Abdullah_Al_Rafi.pdf, page 1
      PUBLICATION
Use of Social Media in Flood Assessment in Bangladesh North South University
IEEE 11th International Conference on Intelligent Systems (IS...

  [3] Abdullah_Al_Rafi.pdf, page 0
      Abdullah Al Raﬁ
+49 163 4005423 | 16(b) |

'This document appears to be a resume or CV for Abdullah Al Raﬁ, detailing his professional experience, education, skills, and achievements in the field of machine learning and data science.'

## 9. Interactive Q&A Loop

Type your questions below. Type `quit` to stop.

In [11]:
print("Document QA — type your questions (type 'quit' to stop)\n")

while True:
    question = input("\nYour question: ").strip()
    if not question:
        continue
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    try:
        ask_question(question)
    except Exception as e:
        print(f"Error: {e}")

Document QA — type your questions (type 'quit' to stop)

Q: I have a upcoming project on business analytics. There should be a smart solution that will take user question then run database query in the DB and retrieve the information with insights. Can I hire him?
------------------------------------------------------------

A: Based on the context provided, Abdullah Al Raﬁ has experience in designing internal LLM-powered business intelligence assistants using RAG (Retrieval-Augmented Generation) and other technologies, which aligns with your project requirements. He has the skills and experience to develop a smart solution that can take user questions, run database queries, and retrieve information with insights. Therefore, it seems like a good fit to consider hiring him for your project.

Sources (4 chunks):

  [1] Abdullah_Al_Rafi.pdf, page 0
      Data Science, bKash Limited Dhaka, Bangladesh
• Migrated and productionized Customer Churn, Business Anomaly Detection, and Customer Lif

## 10. Bonus: Save & Load Vector Store

Persist the in-memory vectorstore to disk so you don't need to re-embed documents every session.

In [12]:
# # Save
# INDEX_PATH = "./inmemory_vectorstore.json"
# vectorstore.dump(INDEX_PATH)
# print(f"Vectorstore saved to {INDEX_PATH}")

In [13]:
# # Load (run this instead of sections 4-6 to skip re-embedding)
# required = ["EMBEDDING_MODEL", "HF_TOKEN"]
# missing = [name for name in required if name not in globals()]
# if missing:
#     raise RuntimeError(f"Missing config values: {missing}. Run section 3 first.")

# INDEX_PATH = "./inmemory_vectorstore.json"
# embeddings = HuggingFaceEndpointEmbeddings(
#     model=EMBEDDING_MODEL,
#     huggingfacehub_api_token=HF_TOKEN,
# )

# vectorstore = InMemoryVectorStore.load(INDEX_PATH, embedding=embeddings)
# print("Vectorstore loaded — ready for questions!")